# 02 — Plan Validation

This notebook tests the project's generic VAL-based plan validator.

Before running:
1. Attach the IPC PDDL dataset with **Add Input**.
2. Keep Internet enabled for the first run.
3. If the validation branch has not been merged yet, keep `BRANCH` set to that branch.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Nikhil3654/generalizable-llm-planning.git"
BRANCH = "feature/plan-validation"  # Change to "main" after the branch is merged.

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "generalizable-llm-planning"
FD_DIR = WORKING_DIR / "downward"
VAL_DIR = WORKING_DIR / "VAL"


## 1. Install VAL build dependencies


In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "cmake", "make", "g++", "flex", "bison"],
    check=True,
)
print("Build dependencies ready.")


## 2. Clone or update the project branch


In [ ]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Repository ready:", REPO_DIR)


## 3. Build Fast Downward


In [ ]:
if not FD_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/aibasel/downward.git", str(FD_DIR)],
        check=True,
    )

subprocess.run([sys.executable, "build.py"], cwd=str(FD_DIR), check=True)

FD_EXECUTABLE = FD_DIR / "fast-downward.py"
if not FD_EXECUTABLE.exists():
    raise FileNotFoundError(FD_EXECUTABLE)

print("Fast Downward ready:", FD_EXECUTABLE)


## 4. Build VAL


In [ ]:
if not VAL_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/KCL-Planning/VAL.git", str(VAL_DIR)],
        check=True,
    )

build_script = VAL_DIR / "scripts/linux/build_linux64.sh"
subprocess.run(["chmod", "+x", str(build_script)], check=True)
subprocess.run(["bash", str(build_script)], cwd=str(VAL_DIR), check=True)

VAL_EXECUTABLE = VAL_DIR / "build/linux64/Release/bin/Validate"
if not VAL_EXECUTABLE.exists():
    raise FileNotFoundError(
        f"VAL built but Validate was not found at {VAL_EXECUTABLE}"
    )

print("VAL ready:", VAL_EXECUTABLE)


## 5. Find the attached Blocksworld dataset


In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

matches = list(KAGGLE_INPUT.rglob("blocks-strips-untyped/domain.pddl"))
if not matches:
    raise FileNotFoundError(
        "Could not find blocks-strips-untyped/domain.pddl. "
        "Attach the IPC PDDL dataset using Add Input."
    )

if len(matches) > 1:
    print("Multiple matches found; using the first:")
    for match in matches:
        print(" -", match)

DOMAIN_FILE = matches[0]
INSTANCE_DIR = DOMAIN_FILE.parent / "instances"

def instance_number(path):
    try:
        return int(path.stem.split("-")[-1])
    except ValueError:
        return path.stem

problems = sorted(INSTANCE_DIR.glob("*.pddl"), key=instance_number)

print("Domain:", DOMAIN_FILE)
print("Problems:", len(problems))
print("First problem:", problems[0])


## 6. Import the project planner and validator


In [ ]:
from src.planner import run_fast_downward, read_plan
from src.validator import validate_plan, write_plan

print("Imports loaded.")


## 7. Generate one correct reference plan


In [ ]:
PROBLEM_FILE = problems[0]
REFERENCE_PLAN = WORKING_DIR / "reference_plan.txt"

if REFERENCE_PLAN.exists():
    REFERENCE_PLAN.unlink()

planner_result = run_fast_downward(
    fast_downward_path=FD_EXECUTABLE,
    domain_file=DOMAIN_FILE,
    problem_file=PROBLEM_FILE,
    plan_file=REFERENCE_PLAN,
)

actions = read_plan(REFERENCE_PLAN)

print("Problem:", PROBLEM_FILE.name)
print("Plan found:", planner_result["plan_found"])
print("Plan length:", len(actions))

if not planner_result["plan_found"]:
    raise RuntimeError("Fast Downward did not produce a reference plan.")


## 8. Validate the correct plan


In [ ]:
correct_result = validate_plan(
    validator_path=VAL_EXECUTABLE,
    domain_file=DOMAIN_FILE,
    problem_file=PROBLEM_FILE,
    plan_file=REFERENCE_PLAN,
)

print(correct_result)
assert correct_result["valid"]


## 9. Create deliberately changed plans


In [ ]:
TRUNCATED_PLAN = WORKING_DIR / "truncated_plan.txt"
UNKNOWN_ACTION_PLAN = WORKING_DIR / "unknown_action_plan.txt"
EMPTY_PLAN = WORKING_DIR / "empty_plan.txt"
SWAPPED_PLAN = WORKING_DIR / "swapped_plan.txt"

write_plan(actions[:-1], TRUNCATED_PLAN)
write_plan(["(not-a-real-action x)"] + actions, UNKNOWN_ACTION_PLAN)
write_plan([], EMPTY_PLAN)

swapped_actions = actions.copy()
if len(swapped_actions) >= 2:
    swapped_actions[0], swapped_actions[1] = swapped_actions[1], swapped_actions[0]
write_plan(swapped_actions, SWAPPED_PLAN)

print("Changed plans written.")


## 10. Validate all variants


In [ ]:
import pandas as pd

variants = {
    "correct": REFERENCE_PLAN,
    "truncated": TRUNCATED_PLAN,
    "unknown_action": UNKNOWN_ACTION_PLAN,
    "empty": EMPTY_PLAN,
    "swapped_first_two": SWAPPED_PLAN,
}

rows = []

for name, plan_path in variants.items():
    result = validate_plan(
        validator_path=VAL_EXECUTABLE,
        domain_file=DOMAIN_FILE,
        problem_file=PROBLEM_FILE,
        plan_file=plan_path,
    )

    rows.append({
        "variant": name,
        "valid": result["valid"],
        "status": result["status"],
        "returncode": result["returncode"],
    })

validation_df = pd.DataFrame(rows)
validation_df


## 11. Save results


In [ ]:
RESULT_PATH = WORKING_DIR / "blocksworld_validation_test.csv"
validation_df.to_csv(RESULT_PATH, index=False)

print(validation_df.to_string(index=False))
print("\nSaved:", RESULT_PATH)
